In [ ]:
import os, sys, joblib
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(".."))

from evaluation.utils.evaluation_functions import load_and_prepare_data_xgb, load_and_prepare_data_baseline
from evaluation.utils.precompute_utils import (
    CurveParams, ShapParams,
    precompute_curves, load_curves_cache,
    precompute_shap_tree, load_shap_cache, shap_explanation_from_cache
)
from evaluation.utils.plotting_utils import (
    set_plot_defaults,
    plot_roc_from_cache, plot_auprc_from_cache,
    plot_calibration_from_cache, plot_decision_curve_from_cache,
    plot_shap_beeswarm_from_cache
)

from sqlalchemy import create_engine
DATABASE_URI = "postgresql+psycopg2://USER@localhost:5434/mimic"

/user/rirg2545/miniconda3/envs/hypotension/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Config

In [2]:
# Optional: set global plot defaults (affects all downstream matplotlib plots)
set_plot_defaults(dpi=600, fonts=dict(title=16, label=26, tick=22, legend=21, shap_label=20, shap_tick=16))

CALIBRATED_MODELS_DIR = "YOUR-PATH/actionable-hypotension/models_given/calibrated"
FIGURES_DIR = "YOUR-PATH/actionable-hypotension/figures/subgroupMAP6575"

### Precompute cache for specific model `class_name`

In [3]:
class_name = "baseline" # <-- select model

In [4]:
# Load validation and test splits from the DB (your function)
x_val, y_val, x_test, y_test, feat_cols = None, None, None, None, None
if class_name == "baseline":
    x_val, y_val, x_test, y_test, feat_cols = load_and_prepare_data_baseline()
else:
    x_val, y_val, x_test, y_test, feat_cols = load_and_prepare_data_xgb(
        table_name=f"merged_{class_name}_features"
    )
    
y_val  = np.asarray(y_val).ravel()
y_test = np.asarray(y_test).ravel()

In [5]:
# Load calibrated model
if class_name == "baseline":
    model = joblib.load(os.path.join(CALIBRATED_MODELS_DIR, "baseline_lr.pkl"))
else:
    model = joblib.load(os.path.join(CALIBRATED_MODELS_DIR, f"xgb_{class_name}.pkl"))

# Use positive-class probabilities
y_val_pred  = model.predict_proba(x_val)
y_test_pred = model.predict_proba(x_test)
y_val_scores  = y_val_pred[:, 1] if y_val_pred.ndim == 2 else y_val_pred
y_test_scores = y_test_pred[:, 1] if y_test_pred.ndim == 2 else y_test_pred

/user/rirg2545/miniconda3/envs/hypotension/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/user/rirg2545/miniconda3/envs/hypotension/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator IsotonicRegression from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# -------- Precompute (heavy) once; later just reload and re-plot --------
curves = precompute_curves(
    y_true=y_test,
    y_prob=y_test_scores,
    class_name=class_name,
    params=CurveParams(
        do_bootstrap=True,          # faster while iterating; enable for final plots
        bootstrap_iterations=1000,
        bootstrap_alpha=0.95,
        cal_n_bins=10,
        cal_strategy="quantile",
        dca_thresholds=None,
        dca_smoothing_sigma=1.0
    ),
    force=True                      # set True to force a refresh even if cache exists
)

In [7]:
# SHAP precompute
model_for_shap = getattr(model, "booster", getattr(model, "booster_", model))
shap_cache = precompute_shap_tree(
    model=model_for_shap,
    X=x_test,
    class_name=class_name,
    params=ShapParams(n_samples=5000, store_X=True, random_state=42),
    force=True
)

ValueError: could not convert string to float: '[1.8333052E-3]'

### Plot plots for specific model `class_name` from precomputed cache

In [8]:
class_name = "mix" # <-- select model

In [7]:
curves = load_curves_cache(class_name)   # latest curves package
plot_roc_from_cache(class_name, curves, out_dir=FIGURES_DIR, xlim=(0, 1), ylim=(0, 1), fig_size=(7,5), show_random=True, save_pdf=True)
plot_auprc_from_cache(class_name, curves, out_dir=FIGURES_DIR, fig_size=(7,5), save_pdf=True)
plot_calibration_from_cache(class_name, curves, out_dir=FIGURES_DIR, fig_size=(7,5), xlim=(0, 0.016), ylim=(0, 0.016), save_pdf=True)
plot_decision_curve_from_cache(class_name, curves, out_dir=FIGURES_DIR, fig_size=(7,5), xlim=(0, 0.06), ylim=(0, 0.0025), save_pdf=True)

'YOUR-PATH/actionable-hypotension/figures/subgroupMAP6575/baseline/DecisionCurve_baseline.png'

In [25]:
shap_cache = load_shap_cache(class_name)
exp = shap_explanation_from_cache(shap_cache)
plot_shap_beeswarm_from_cache(class_name + "_full", exp, out_dir=FIGURES_DIR, fig_size=(6,4), max_display=37, save_pdf=True)

'figures/noninv_full/SHAP_noninv_full.png'